# Configs

In [1]:
import pandas as pd
import numpy as np

In [2]:
Proj_path = '/Users/cmartin/Fantasy_Baseball/Ottoneu_Baseball_Projects/FG_2026_Projections'

In [3]:
Player_ID_Map_Path = '/Users/cmartin/Fantasy_Baseball/Ottoneu_Baseball_Projects/Player_ID_Map/Latest_Player_ID_Map.csv'

In [4]:
Proj_date = 'Mar21_2026'

In [5]:
Projections_dates = {
    'ATC':'Mar21_2026',
    'OOPSY':'Mar21_2026'
}

In [6]:
Proj_weights = {
    'ATC':.6,
    'OOPSY':.4
}

In [7]:
Hitting_Possitions = [
    'C',
    '1B',
    '2B',
    'SS',
    '3B',
    'OF',
    'DH'
]
Pitching_Possitions = [
    'SP',
    'RP'
]

In [8]:
Player_id_cols = [
    'FG ID', 'Team'#,'MLBAMID','Name','Team','NameASCII'
]
Hitter_Count_Stats = [
    'G','PA','AB','H','1B','2B','3B','HR','R','RBI','BB','HBP','SF','WAR','ADP'
]
Pitcher_Count_Stats = [
    'W', 'L', 'QS', 'G', 'GS', 'SV', 'HLD', 'IP', 'TBF', 'H', 'R', 'ER', 'HR', 'BB', 'HBP', 'SO','WAR','ADP'
] # 'BS',

In [9]:
def w_avg(df, values, weights):
    d = df[values]
    w = df[weights]
    return (d * w).sum() / w.sum()

In [10]:
Hitter_Projections_df = pd.DataFrame()
for proj,date in Projections_dates.items():
    this_proj_df = pd.DataFrame()
    for pos in Hitting_Possitions:
        temp_df = pd.read_csv(f"{Proj_path}/{date}/{proj}/fangraphs-{proj}_{date}-projections_{pos}.csv")
        temp_df['Projection'] = proj
        temp_df['Proj_weight'] = Proj_weights[proj]
        temp_df['POS'] = pos
        temp_df.rename(columns={
            'PlayerId':'FG ID'
        },inplace=True)
        temp_df['FG ID'] = temp_df['FG ID'].astype(str)
        temp_df['FG ID'] = temp_df['FG ID'].str.replace('.0','')
        this_proj_df = pd.concat([this_proj_df, temp_df], ignore_index=True)
    this_proj_df = this_proj_df[Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                this_proj_df.groupby(Player_id_cols+Hitter_Count_Stats)['POS'].agg(list).reset_index()
        )
    Hitter_Projections_df = pd.concat([Hitter_Projections_df,this_proj_df], ignore_index=True)
Hitter_Projections_df = Hitter_Projections_df[Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                Hitter_Projections_df.groupby(Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight'])['POS'].agg(list).reset_index()
        )


Pitcher_Projections_df = pd.DataFrame()
for proj,date in Projections_dates.items():
    this_proj_df = pd.DataFrame()
    for pos in Pitching_Possitions:
        temp_df = pd.read_csv(f"{Proj_path}/{date}/{proj}/fangraphs-{proj}_{date}-projections_{pos}.csv")
        temp_df['Projection'] = proj
        temp_df['Proj_weight'] = Proj_weights[proj]
        temp_df['POS'] = pos
        temp_df.rename(columns={
            'PlayerId':'FG ID'
        },inplace=True)
        temp_df['FG ID'] = temp_df['FG ID'].astype(str)
        temp_df['FG ID'] = temp_df['FG ID'].str.replace('.0','')
        this_proj_df = pd.concat([this_proj_df, temp_df], ignore_index=True)
    this_proj_df = this_proj_df[Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                this_proj_df.groupby(Player_id_cols+Pitcher_Count_Stats)['POS'].agg(list).reset_index()
        )
    Pitcher_Projections_df = pd.concat([Pitcher_Projections_df,this_proj_df], ignore_index=True)
Pitcher_Projections_df = Pitcher_Projections_df[Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                Pitcher_Projections_df.groupby(Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight'])['POS'].agg(list).reset_index()
        )

In [11]:
for col in Player_id_cols:
    Hitter_Projections_df[col] = Hitter_Projections_df[col].astype(str)
    Pitcher_Projections_df[col] = Pitcher_Projections_df[col].astype(str)

In [12]:
Player_ID_Map_df = pd.read_csv(Player_ID_Map_Path)
Player_ID_Map_df['FG ID'] = Player_ID_Map_df['FG ID'].astype(str)
Player_ID_Map_df['FG ID'] = Player_ID_Map_df['FG ID'].fillna(Player_ID_Map_df['FG Minor ID'])
Player_ID_Map_df['FG ID'] = Player_ID_Map_df['FG ID'].str.replace('.0','')
Player_ID_Map_df['Ottoneu ID'] = Player_ID_Map_df['Ottoneu ID'].astype(str)
Player_ID_Map_df['Ottoneu ID'] = Player_ID_Map_df['Ottoneu ID'].str.replace('.0','')

In [13]:
Pitcher_Projections_df.head()

,FG ID,Team,W,L,QS,G,GS,SV,HLD,IP,...,ER,HR,BB,HBP,SO,WAR,ADP,Projection,Proj_weight,POS
0,22267,DET,14.2544,6.59444,18.1613,29.4535,29.4535,0.0,0.000000,188.890,...,56.9220,18.3742,38.3067,6.41576,228.693,5.91247,6.880000,ATC,0.6,[[SP]]
1,33677,PIT,12.0813,7.81657,17.5098,29.8913,29.8913,0.0,0.000000,185.006,...,56.6058,15.2685,45.7703,6.45747,221.408,5.69184,9.820000,ATC,0.6,[[SP]]
2,27463,BOS,14.0367,8.01748,16.6665,29.3401,29.3401,0.0,0.000000,183.850,...,62.1955,19.6361,47.4583,5.17476,225.430,5.11006,11.950000,ATC,0.6,[[SP]]
3,17995,SFG,13.2676,9.54779,17.0111,30.0462,30.0462,0.0,0.177689,194.426,...,71.2688,15.4999,44.1717,5.82888,184.151,4.86479,57.950001,ATC,0.6,[[SP]]
4,20778,PHI,12.0397,7.50996,16.6067,29.3479,29.3479,0.0,0.000000,187.813,...,66.5356,16.1629,45.7879,5.90043,182.578,4.59634,25.650000,ATC,0.6,[[SP]]


In [14]:
Pitcher_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Pitcher_Count_Stats], weights=x["Proj_weight"], axis=0), Pitcher_Count_Stats))

,,W,L,QS,G,GS,SV,HLD,IP,TBF,H,R,ER,HR,BB,HBP,SO,WAR,ADP
FG ID,Team,,,,,,,,,,,,,,,,,,
10061,NYM,3.457276,2.612226,0.00000,57.41750,0.00000,1.432636,14.970260,57.17894,239.0132,48.83716,25.48488,23.51008,6.902372,19.88426,3.268920,57.27418,0.374271,999.000000
10078,CHC,3.810766,3.090756,0.00000,63.23826,0.00000,1.313236,17.114160,61.95228,255.9154,52.67776,27.65996,25.57512,7.719272,20.94138,2.039232,63.77168,0.433626,999.000000
10197,CHC,0.000000,0.000000,0.00000,1.00000,0.00000,0.000000,0.000000,1.01771,4.0000,1.00000,0.00000,0.00000,0.000000,0.00000,0.000000,1.00000,0.005986,999.000000
10233,BOS,4.411340,3.169422,0.00000,58.36838,0.00000,29.244220,2.260412,58.20950,238.2870,40.86454,20.35538,18.96694,5.019878,25.46314,2.337930,80.99644,1.200228,64.040001
10310,PHI,10.372512,6.042328,14.16896,22.88546,22.88546,0.000000,0.000000,141.50760,573.0358,115.41600,55.41562,50.93438,16.712800,38.46164,6.173578,165.16580,3.290876,113.110001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
sa918182,CIN,0.000000,0.000000,0.00000,1.00000,1.00000,0.000000,0.000000,5.33333,20.0000,4.00000,3.00000,3.00000,1.000000,2.00000,0.000000,5.00000,0.081386,999.000000
sa920454,PHI,0.000000,0.000000,0.00000,1.00000,0.00000,0.000000,0.000000,1.18531,5.0000,1.00000,1.00000,1.00000,0.000000,1.00000,0.000000,1.00000,-0.001062,999.000000
sa928287,MIN,0.000000,0.000000,0.00000,1.00000,0.00000,0.000000,0.000000,1.55512,7.0000,1.00000,1.00000,1.00000,0.000000,1.00000,0.000000,2.00000,0.000814,999.000000


In [15]:
Hitter_weighted_averages = Hitter_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Hitter_Count_Stats], weights=x["Proj_weight"], axis=0), Hitter_Count_Stats)).reset_index()
Pitcher_weighted_averages = Pitcher_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Pitcher_Count_Stats], weights=x["Proj_weight"], axis=0), Pitcher_Count_Stats)).reset_index()

In [16]:
Hitter_weighted_averages.shape[0]

2341

In [17]:
Pitcher_weighted_averages.shape[0]

2945

In [18]:
Hitter_Projections_df['POS'] = Hitter_Projections_df['POS'].apply(lambda pos_list_of_lists: pos_list_of_lists[0])
Pitcher_Projections_df['POS'] = Pitcher_Projections_df['POS'].apply(lambda pos_list_of_lists: pos_list_of_lists[0])

In [19]:
Hitter_pos_merged = Hitter_Projections_df.groupby(Player_id_cols)['POS'].apply(lambda pos_list: list(set(sum(pos_list,[])))).reset_index()
Pitcher_pos_merged = Pitcher_Projections_df.groupby(Player_id_cols)['POS'].apply(lambda pos_list: list(set(sum(pos_list,[])))).reset_index()

In [20]:
Hitter_pos_merged.shape[0]

2341

In [21]:
Pitcher_pos_merged.shape[0]

2945

In [22]:
Hitter_final_merge_df=Hitter_weighted_averages.merge(Hitter_pos_merged,on=Player_id_cols,how='outer')
Pitcher_final_merge_df=Pitcher_weighted_averages.merge(Pitcher_pos_merged,on=Player_id_cols,how='outer')

In [23]:
Hitter_final_merge_df = Hitter_final_merge_df.merge(Player_ID_Map_df[['FG ID','Ottoneu ID','Ottoneu Positions','Name']],how='inner')
Pitcher_final_merge_df = Pitcher_final_merge_df.merge(Player_ID_Map_df[['FG ID','Ottoneu ID','Ottoneu Positions','Name']],how='inner')

In [24]:
# Find Duplicate Players
Hitter_final_merge_df[Hitter_final_merge_df['Ottoneu ID'].isin([key for key, val in Hitter_final_merge_df['Ottoneu ID'].value_counts().to_dict().items() if val > 1])].sort_values('Name')

,FG ID,Team,G,PA,AB,H,1B,2B,3B,HR,...,RBI,BB,HBP,SF,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name


In [25]:
Pitcher_final_merge_df[Pitcher_final_merge_df['Ottoneu ID'].isin([key for key, val in Pitcher_final_merge_df['Ottoneu ID'].value_counts().to_dict().items() if val > 1])].sort_values('Name')

,FG ID,Team,W,L,QS,G,GS,SV,HLD,IP,...,HR,BB,HBP,SO,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name


In [26]:
Hitter_final_merge_df.head()

,FG ID,Team,G,PA,AB,H,1B,2B,3B,HR,...,RBI,BB,HBP,SF,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name
0,10028,CHC,0.00000,1.0000,1.0000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.001407,999.000000,[C],18048,C,Christian Bethancourt
1,10155,LAA,122.63920,525.3260,444.6746,106.01270,60.76846,17.718100,1.799282,25.726840,...,68.54890,70.244320,6.435478,3.700210,2.555656,178.910004,"[DH, OF]",6305,OF,Mike Trout
2,10243,NYY,35.85476,144.5400,132.6196,31.55082,19.18724,6.811464,0.400766,5.551346,...,17.61502,9.106418,1.220332,1.145086,0.188103,999.000000,"[DH, OF]",6320,OF,Randal Grichuk
3,10324,PIT,123.24320,522.2472,453.2730,110.13880,69.28562,19.842960,0.108619,20.901560,...,66.68648,61.832180,3.001938,4.031374,0.927671,372.309998,"[DH, OF]",6130,Util,Marcell Ozuna
4,10472,LAD,47.12238,186.0596,169.1838,37.58262,24.30588,8.091872,0.086417,5.498438,...,21.55730,13.425500,1.007362,1.602070,0.117489,999.000000,"[1B, OF]",11508,1B/2B/3B/OF/RP,Enrique Hernandez


In [27]:
# Fix known Pitcher Hitter overlaps, manually choosing which position for these players.
# Note: This explicitly leaves some players as duplicates because they are 2-way players. (e.g. Otahni)

PlayerIds_drop_Pitching = []
PlayerIds_drop_Hitting = []

# Andrés Nolaya sa3018563
PlayerIds_drop_Pitching.append('sa3018563')

# Ben Malgeri sa3016975
PlayerIds_drop_Pitching.append('sa3016975')

# Caleb Pendleton sa3022483
PlayerIds_drop_Pitching.append('sa3022483')

# Carlos Franco sa3015580
PlayerIds_drop_Pitching.append('sa3015580')

# Chase Valentine sa3020002
PlayerIds_drop_Pitching.append('sa3020002')

# Curtis Washington Jr. sa3019803	
PlayerIds_drop_Pitching.append('sa3019803')

# David Leon sa3016336
PlayerIds_drop_Pitching.append('sa3016336')

# Diego Viloria sa3016299
PlayerIds_drop_Hitting.append('sa3016299')

# Fernando Caldera sa3015547
PlayerIds_drop_Pitching.append('sa3015547')

# Jackson Cluff sa3010342
PlayerIds_drop_Hitting.append('sa3010342')

# Jesus Castro sa3019090
PlayerIds_drop_Hitting.append('sa3019090')

# Johan Contreras sa3019217
PlayerIds_drop_Pitching.append('sa3019217')

# Jose Gerardo sa3018645
PlayerIds_drop_Hitting.append('sa3018645')

# Keduar Trujillo sa3019764
PlayerIds_drop_Pitching.append('sa3019764')

# Sean Barnett sa3025496
PlayerIds_drop_Hitting.append('sa3025496')

# Wyatt Hoffman sa3018219
PlayerIds_drop_Pitching.append('sa3018219')

# Yolmer Sánchez 11602
PlayerIds_drop_Pitching.append('11602')

# Anthony Prato	sa1052903
PlayerIds_drop_Pitching.append('sa1052903')

# Nick Kahle sa1115979
PlayerIds_drop_Pitching.append('sa1115979')

# Dylan Shockley sa1170874
PlayerIds_drop_Pitching.append('sa1170874')

# Andy Weber sa3008282
PlayerIds_drop_Pitching.append('sa3008282')

# Maikol Escotto sa3008878
PlayerIds_drop_Pitching.append('sa3008878')

# Myles Emmerson sa3016862
PlayerIds_drop_Pitching.append('sa3016862')

# Cole Urman sa3023270
PlayerIds_drop_Pitching.append('sa3023270')

In [28]:
Pitcher_final_merge_df = Pitcher_final_merge_df[~Pitcher_final_merge_df['FG ID'].isin(PlayerIds_drop_Pitching)]
Hitter_final_merge_df = Hitter_final_merge_df[~Hitter_final_merge_df['FG ID'].isin(PlayerIds_drop_Hitting)]

In [29]:
Hitter_final_merge_df.head()

,FG ID,Team,G,PA,AB,H,1B,2B,3B,HR,...,RBI,BB,HBP,SF,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name
0,10028,CHC,0.00000,1.0000,1.0000,0.00000,0.00000,0.000000,0.000000,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.001407,999.000000,[C],18048,C,Christian Bethancourt
1,10155,LAA,122.63920,525.3260,444.6746,106.01270,60.76846,17.718100,1.799282,25.726840,...,68.54890,70.244320,6.435478,3.700210,2.555656,178.910004,"[DH, OF]",6305,OF,Mike Trout
2,10243,NYY,35.85476,144.5400,132.6196,31.55082,19.18724,6.811464,0.400766,5.551346,...,17.61502,9.106418,1.220332,1.145086,0.188103,999.000000,"[DH, OF]",6320,OF,Randal Grichuk
3,10324,PIT,123.24320,522.2472,453.2730,110.13880,69.28562,19.842960,0.108619,20.901560,...,66.68648,61.832180,3.001938,4.031374,0.927671,372.309998,"[DH, OF]",6130,Util,Marcell Ozuna
4,10472,LAD,47.12238,186.0596,169.1838,37.58262,24.30588,8.091872,0.086417,5.498438,...,21.55730,13.425500,1.007362,1.602070,0.117489,999.000000,"[1B, OF]",11508,1B/2B/3B/OF/RP,Enrique Hernandez


In [30]:
#Test Hitter / Pitcher Concat
Quick_test = pd.concat([Hitter_final_merge_df,Pitcher_final_merge_df])
Quick_test[Quick_test['Ottoneu ID'].isin([key for key, val in Quick_test['Ottoneu ID'].value_counts().to_dict().items() if val > 1])][['Name','Ottoneu ID','FG ID','PA','IP']]

,Name,Ottoneu ID,FG ID,PA,IP
217,Shohei Ohtani,33600,19755,667.687,NaN
1189,Santiago Gomez,41367,sa3018616,1.000,NaN
1727,Sheng-En Lin,43442,sa3022820,1.000,NaN
2327,Riley Unroe,19282,sa737614,1.000,NaN
292,Shohei Ohtani,33600,19755,NaN,113.53200
1469,Santiago Gomez,41367,sa3018616,NaN,0.65721
2094,Sheng-En Lin,43442,sa3022820,NaN,3.08667
2914,Riley Unroe,19282,sa737614,NaN,1.51030


In [31]:
Hitter_final_merge_df.to_csv(Proj_path+f'/my_Hitter_Proj_{Proj_date}.csv',index=False)

In [32]:
Pitcher_final_merge_df.to_csv(Proj_path+f'/my_Pitcher_Proj_{Proj_date}.csv',index=False)